### Cell 1 — Setup

In [5]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

import shap

from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

DATA_DIR = Path('../data/processed')
RESULTS_DIR = Path('../results')
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
MODELS_DIR = RESULTS_DIR / 'models'

RANDOM_STATE = 42

# Load encoded permissive data
df_encoded = pd.read_csv(DATA_DIR / 'metabric_permissive_encoded.csv')

# Load split indices
splits = pd.read_csv(DATA_DIR / 'split_indices.csv')
train_idx = splits['permissive_train'].dropna().astype(int).values
test_idx = splits['permissive_test'].dropna().astype(int).values

X = df_encoded.drop(columns=['Relapse Free Status']).astype(float)
y = df_encoded['Relapse Free Status'].astype(int)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Reset index on test set so we can refer to patients by 0-based position
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Load the canonical GBM
with open(MODELS_DIR / 'canonical_gbm.pkl', 'rb') as f:
    gbm_data = pickle.load(f)
gbm = gbm_data['model']

# Verify GBM still produces Phase 2 numbers
gbm_pred = gbm.predict(X_test)
gbm_probs = gbm.predict_proba(X_test)[:, 1]
print(f"Canonical GBM (reload check):")
print(f"  Test accuracy: {accuracy_score(y_test, gbm_pred):.3f}")
print(f"  Test recall:   {recall_score(y_test, gbm_pred, pos_label=1):.3f}")
print(f"\nTest set: {len(X_test)} patients")

Canonical GBM (reload check):
  Test accuracy: 0.665
  Test recall:   0.344

Test set: 394 patients


### Cell 2 — Compute SHAP values

In [6]:
# === Compute SHAP values via TreeExplainer ===

print("Building TreeExplainer for canonical GBM...")
explainer = shap.TreeExplainer(gbm)

print("Computing SHAP values for test set...")
# For binary classification, this returns SHAP values for the positive class
shap_values = explainer.shap_values(X_test)

# Some sklearn GBM versions return a 3D array (n_samples, n_features, n_classes);
# others return 2D. Handle both.
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]  # take positive-class values

# Sanity check: SHAP values should sum (with the base value) to the model's log-odds output
# Just confirm shapes
print(f"\nSHAP values shape: {shap_values.shape}")
print(f"Test set X shape:  {X_test.shape}")
print(f"Expected: ({len(X_test)}, {X_test.shape[1]})")

# Compute mean absolute SHAP value per feature for global importance
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=X.columns).sort_values(ascending=False)

print("\nTop 15 features by mean |SHAP value|:")
for feat, val in shap_importance.head(15).items():
    print(f"  {val:.4f}  {feat}")

Building TreeExplainer for canonical GBM...
Computing SHAP values for test set...

SHAP values shape: (394, 39)
Test set X shape:  (394, 39)
Expected: (394, 39)

Top 15 features by mean |SHAP value|:
  0.2744  Lymph nodes examined positive
  0.1418  Tumor Size(mm)
  0.0873  Nottingham prognostic index
  0.0733  Age at Diagnosis
  0.0496  Pam50 + Claudin-low subtype_LumB
  0.0333  Pam50 + Claudin-low subtype_LumA
  0.0290  Mutation Count
  0.0220  G-Neoplasm Histologic Grade
  0.0164  HER2 Status_Negative
  0.0152  HER2 Status_Positive
  0.0083  Cellularity
  0.0077  Hormone Therapy_Yes
  0.0065  Hormone Therapy_No
  0.0054  Pam50 + Claudin-low subtype_Her2
  0.0049  Pam50 + Claudin-low subtype_claudin-low


### Cell 3 — Compare SHAP importance to the GBM's built-in feature importance

In [7]:
# === Compare SHAP importance vs GBM built-in importance ===

gbm_importance = pd.Series(gbm.feature_importances_, index=X.columns).sort_values(ascending=False)

# Build a side-by-side comparison
top_n = 15
comparison = pd.DataFrame({
    'gbm_importance': gbm_importance,
    'shap_importance': shap_importance,
})
comparison = comparison.sort_values('gbm_importance', ascending=False).head(top_n)

# Add ranks for easy comparison
comparison['gbm_rank'] = comparison['gbm_importance'].rank(ascending=False).astype(int)
comparison['shap_rank'] = comparison['shap_importance'].rank(ascending=False).astype(int)

print("Top features by GBM importance, with SHAP rank shown:")
print(comparison[['gbm_importance', 'gbm_rank', 'shap_importance', 'shap_rank']].round(4))

Top features by GBM importance, with SHAP rank shown:
                                         gbm_importance  gbm_rank  \
Lymph nodes examined positive                    0.4022         1   
Nottingham prognostic index                      0.1479         2   
Tumor Size(mm)                                   0.1392         3   
Age at Diagnosis                                 0.1012         4   
Mutation Count                                   0.0445         5   
Pam50 + Claudin-low subtype_LumB                 0.0333         6   
Pam50 + Claudin-low subtype_LumA                 0.0286         7   
G-Neoplasm Histologic Grade                      0.0188         8   
HER2 Status_Negative                             0.0167         9   
Cellularity                                      0.0151        10   
HER2 Status_Positive                             0.0147        11   
Hormone Therapy_No                               0.0052        12   
Tumor Other Histologic Subtype_Mixed             

### Cell 4 — Identify the three patient categories

In [8]:
# === Identify three patient categories ===

# Compute prediction confidence: distance from 0.5
confidence = np.abs(gbm_probs - 0.5)

# Categorize
categories = pd.DataFrame({
    'patient_idx': range(len(X_test)),
    'true_label': y_test.values,
    'predicted_prob': gbm_probs,
    'predicted_label': gbm_pred,
    'confidence': confidence,
    'correct': (gbm_pred == y_test.values),
})

# Confident threshold: top quartile of confidence
high_conf_threshold = np.quantile(confidence, 0.75)
borderline_threshold = 0.05  # within 0.05 of 0.5

print(f"Confidence threshold (top quartile): |prob - 0.5| >= {high_conf_threshold:.3f}")
print(f"Borderline threshold: |prob - 0.5| < {borderline_threshold}")
print()

confident_correct = categories[(categories['confidence'] >= high_conf_threshold) & categories['correct']]
confident_wrong = categories[(categories['confidence'] >= high_conf_threshold) & ~categories['correct']]
borderline = categories[categories['confidence'] < borderline_threshold]

print(f"Confident-correct patients: {len(confident_correct)}")
print(f"Confident-wrong patients:   {len(confident_wrong)}")
print(f"Borderline patients:        {len(borderline)}")

# Show the breakdown by true class
print(f"\nConfident-wrong breakdown:")
print(f"  False positives (predicted recurrence, didn't recur): "
      f"{((confident_wrong['predicted_label'] == 1) & (confident_wrong['true_label'] == 0)).sum()}")
print(f"  False negatives (missed recurrences): "
      f"{((confident_wrong['predicted_label'] == 0) & (confident_wrong['true_label'] == 1)).sum()}")

Confidence threshold (top quartile): |prob - 0.5| >= 0.192
Borderline threshold: |prob - 0.5| < 0.05

Confident-correct patients: 77
Confident-wrong patients:   22
Borderline patients:        59

Confident-wrong breakdown:
  False positives (predicted recurrence, didn't recur): 3
  False negatives (missed recurrences): 19


### Cell 5 — Compare SHAP feature usage across categories

In [9]:
# === Compare which features SHAP highlights across categories ===

def top_features_for_category(category_indices, n_top=5):
    """Return the top N features by mean |SHAP| for the given patients."""
    if len(category_indices) == 0:
        return pd.Series(dtype=float)
    cat_shap = shap_values[category_indices]
    mean_abs = np.abs(cat_shap).mean(axis=0)
    return pd.Series(mean_abs, index=X.columns).sort_values(ascending=False).head(n_top)


print("="*60)
print("  SHAP FEATURE USAGE BY CATEGORY")
print("="*60)

print("\nConfident-correct patients — top features:")
print(top_features_for_category(confident_correct['patient_idx'].values).round(4))

print("\nConfident-wrong patients — top features:")
print(top_features_for_category(confident_wrong['patient_idx'].values).round(4))

print("\nBorderline patients — top features:")
print(top_features_for_category(borderline['patient_idx'].values).round(4))

  SHAP FEATURE USAGE BY CATEGORY

Confident-correct patients — top features:
Lymph nodes examined positive    0.3003
Nottingham prognostic index      0.1478
Tumor Size(mm)                   0.1406
Age at Diagnosis                 0.0880
G-Neoplasm Histologic Grade      0.0425
dtype: float64

Confident-wrong patients — top features:
Lymph nodes examined positive       0.3092
Tumor Size(mm)                      0.1367
Nottingham prognostic index         0.1265
Age at Diagnosis                    0.1037
Pam50 + Claudin-low subtype_LumB    0.0441
dtype: float64

Borderline patients — top features:
Lymph nodes examined positive       0.3137
Tumor Size(mm)                      0.1481
Nottingham prognostic index         0.0674
Age at Diagnosis                    0.0518
Pam50 + Claudin-low subtype_LumB    0.0505
dtype: float64


### Cell 6 — Drill into specific confident-wrong patients

In [10]:
# === Drill into specific confident-wrong patients ===

# Sort by confidence (most confident-and-wrong first)
confident_wrong_sorted = confident_wrong.sort_values('confidence', ascending=False).head(3)

print("="*60)
print("  CASE STUDIES: confident-wrong patients")
print("="*60)
print()

for _, row in confident_wrong_sorted.iterrows():
    idx = row['patient_idx']
    
    print(f"--- Patient at index {idx} ---")
    print(f"  Predicted: {'RECURRENCE' if row['predicted_label'] == 1 else 'NO RECURRENCE'} "
          f"(P = {row['predicted_prob']:.3f})")
    print(f"  Actual:    {'RECURRENCE' if row['true_label'] == 1 else 'NO RECURRENCE'}")
    print(f"  Confidence: {row['confidence']:.3f}")
    
    # Top contributing features
    patient_shap = shap_values[int(idx)]
    patient_features = X_test.iloc[int(idx)]
    
    contributions = pd.DataFrame({
        'feature': X.columns,
        'value': patient_features.values,
        'shap': patient_shap,
        'abs_shap': np.abs(patient_shap),
    }).sort_values('abs_shap', ascending=False).head(8)
    
    print("  Top 8 SHAP contributions:")
    for _, c in contributions.iterrows():
        direction = "↑ recurrence" if c['shap'] > 0 else "↓ recurrence"
        print(f"    {c['feature']}: value={c['value']:.2f}, "
              f"SHAP={c['shap']:+.3f} ({direction})")
    print()

  CASE STUDIES: confident-wrong patients

--- Patient at index 26 ---
  Predicted: NO RECURRENCE (P = 0.234)
  Actual:    RECURRENCE
  Confidence: 0.266
  Top 8 SHAP contributions:
    Nottingham prognostic index: value=-1.72, SHAP=-0.227 (↓ recurrence)
    Lymph nodes examined positive: value=-0.47, SHAP=-0.172 (↓ recurrence)
    Tumor Size(mm): value=-0.63, SHAP=-0.155 (↓ recurrence)
    G-Neoplasm Histologic Grade: value=-2.25, SHAP=-0.141 (↓ recurrence)
    Age at Diagnosis: value=-0.62, SHAP=+0.046 (↑ recurrence)
    Pam50 + Claudin-low subtype_LumA: value=1.00, SHAP=-0.038 (↓ recurrence)
    Pam50 + Claudin-low subtype_LumB: value=0.00, SHAP=-0.034 (↓ recurrence)
    Hormone Therapy_No: value=0.00, SHAP=-0.017 (↓ recurrence)

--- Patient at index 312 ---
  Predicted: NO RECURRENCE (P = 0.234)
  Actual:    RECURRENCE
  Confidence: 0.266
  Top 8 SHAP contributions:
    Lymph nodes examined positive: value=-0.47, SHAP=-0.229 (↓ recurrence)
    Nottingham prognostic index: value=-1.7

### Cell 7 — Check the Patient 81 hypothesis directly

In [11]:
# === Quantify the Patient 81 hypothesis ===
# Key claim: confident-wrong explanations are as "coherent" as confident-correct ones,
# meaning the explanation cannot distinguish correct from wrong predictions.
#
# We'll measure this two ways:
# 1. Concentration of SHAP values: are explanations as focused (top-features-driven) for
#    wrong predictions as for correct ones? If yes, an observer can't tell from the
#    explanation whether the model is right.
# 2. Feature overlap: does the same set of features appear at the top of both
#    confident-correct and confident-wrong explanations?

def gini_concentration(shap_row):
    """Gini coefficient on absolute SHAP values — how concentrated is the explanation?"""
    abs_vals = np.sort(np.abs(shap_row))
    n = len(abs_vals)
    if abs_vals.sum() == 0:
        return 0
    index = np.arange(1, n + 1)
    return (2 * (index * abs_vals).sum() - (n + 1) * abs_vals.sum()) / (n * abs_vals.sum())


# Concentration comparison
cc_concentrations = [
    gini_concentration(shap_values[int(i)])
    for i in confident_correct['patient_idx'].values
]
cw_concentrations = [
    gini_concentration(shap_values[int(i)])
    for i in confident_wrong['patient_idx'].values
]
bl_concentrations = [
    gini_concentration(shap_values[int(i)])
    for i in borderline['patient_idx'].values
]

print("Explanation concentration (Gini coefficient on |SHAP|):")
print(f"  Confident-correct:  mean={np.mean(cc_concentrations):.3f}, "
      f"std={np.std(cc_concentrations):.3f}")
print(f"  Confident-wrong:    mean={np.mean(cw_concentrations):.3f}, "
      f"std={np.std(cw_concentrations):.3f}")
print(f"  Borderline:         mean={np.mean(bl_concentrations):.3f}, "
      f"std={np.std(bl_concentrations):.3f}")
print()
print("If confident-correct and confident-wrong have similar concentration,")
print("the explanation alone cannot distinguish them — Patient 81 finding holds.")

# Feature overlap
top_features_cc = set(top_features_for_category(confident_correct['patient_idx'].values, n_top=5).index)
top_features_cw = set(top_features_for_category(confident_wrong['patient_idx'].values, n_top=5).index)
overlap = top_features_cc & top_features_cw
print(f"\nTop 5 features for confident-correct: {top_features_cc}")
print(f"Top 5 features for confident-wrong:   {top_features_cw}")
print(f"Overlap: {len(overlap)} of 5 ({100*len(overlap)/5:.0f}%)")
print(f"Overlapping features: {overlap}")

Explanation concentration (Gini coefficient on |SHAP|):
  Confident-correct:  mean=0.856, std=0.018
  Confident-wrong:    mean=0.854, std=0.021
  Borderline:         mean=0.868, std=0.020

If confident-correct and confident-wrong have similar concentration,
the explanation alone cannot distinguish them — Patient 81 finding holds.

Top 5 features for confident-correct: {'Tumor Size(mm)', 'Lymph nodes examined positive', 'Nottingham prognostic index', 'G-Neoplasm Histologic Grade', 'Age at Diagnosis'}
Top 5 features for confident-wrong:   {'Tumor Size(mm)', 'Lymph nodes examined positive', 'Pam50 + Claudin-low subtype_LumB', 'Nottingham prognostic index', 'Age at Diagnosis'}
Overlap: 4 of 5 (80%)
Overlapping features: {'Lymph nodes examined positive', 'Nottingham prognostic index', 'Tumor Size(mm)', 'Age at Diagnosis'}


### Cell 8 — Generate SHAP plots for the memo

In [12]:
# === Generate SHAP summary plot for the memo ===

# Standard SHAP summary plot — shows feature importance and direction of effect
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved SHAP summary plot to {FIGURES_DIR / 'shap_summary.png'}")

# Bar plot of mean |SHAP| — cleaner for memo if summary plot is too busy
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_importance_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved SHAP bar plot to {FIGURES_DIR / 'shap_importance_bar.png'}")

Saved SHAP summary plot to ..\results\figures\shap_summary.png
Saved SHAP bar plot to ..\results\figures\shap_importance_bar.png


### Cell 9 — Save results table

In [13]:
# === Save SHAP analysis results table ===

results = {
    'n_confident_correct': len(confident_correct),
    'n_confident_wrong': len(confident_wrong),
    'n_borderline': len(borderline),
    'concentration_confident_correct_mean': float(np.mean(cc_concentrations)),
    'concentration_confident_correct_std': float(np.std(cc_concentrations)),
    'concentration_confident_wrong_mean': float(np.mean(cw_concentrations)),
    'concentration_confident_wrong_std': float(np.std(cw_concentrations)),
    'concentration_borderline_mean': float(np.mean(bl_concentrations)),
    'concentration_borderline_std': float(np.std(bl_concentrations)),
    'top5_overlap_pct': 100 * len(overlap) / 5,
}

results_df = pd.DataFrame([results]).round(3)
results_df.to_csv(TABLES_DIR / 'shap_analysis.csv', index=False)
print(f"Saved SHAP analysis summary to {TABLES_DIR / 'shap_analysis.csv'}")

# Also save the SHAP importance ranking for the memo
shap_importance.head(20).to_csv(TABLES_DIR / 'shap_feature_importance.csv',
                                  header=['mean_abs_shap'])
print(f"Saved SHAP feature importance to {TABLES_DIR / 'shap_feature_importance.csv'}")

print()
print("Phase 7 results summary:")
for k, v in results.items():
    print(f"  {k}: {v}")

Saved SHAP analysis summary to ..\results\tables\shap_analysis.csv
Saved SHAP feature importance to ..\results\tables\shap_feature_importance.csv

Phase 7 results summary:
  n_confident_correct: 77
  n_confident_wrong: 22
  n_borderline: 59
  concentration_confident_correct_mean: 0.8559451618364131
  concentration_confident_correct_std: 0.0184535369826872
  concentration_confident_wrong_mean: 0.8537953904252586
  concentration_confident_wrong_std: 0.02082158963785064
  concentration_borderline_mean: 0.8682370512177369
  concentration_borderline_std: 0.019958336212936013
  top5_overlap_pct: 80.0


## Phase 7 Summary: SHAP

### What was tested

SHAP analysis of the canonical GBM (Phase 2 model). Three components:
1. **Global feature importance** via TreeExplainer — comparison to the GBM's built-in feature importance.
2. **Patient categorization** — confident-correct (top quartile of confidence and prediction matches truth), confident-wrong (top quartile and prediction is wrong), borderline (probability within 0.05 of 0.5).
3. **Patient 81 quantification** — concentration (Gini coefficient on |SHAP|) and top-5 feature overlap between confident-correct and confident-wrong groups, plus case studies of the three highest-confidence wrong predictions.

### Headline numbers

| Metric | Confident-Correct | Confident-Wrong | Borderline |
|---|---|---|---|
| N patients | 77 | 22 | 59 |
| Concentration (Gini) | 0.856 ± 0.018 | 0.854 ± 0.021 | 0.868 ± 0.020 |
| Top features (mean \|SHAP\|) | Lymph nodes, NPI, Tumor Size, Age, Grade | Lymph nodes, Tumor Size, NPI, Age, Pam50 LumB | Lymph nodes, Tumor Size, NPI, Age, Pam50 LumB |

**Confident-wrong breakdown**: 19 false negatives (missed recurrences), 3 false positives. The model's confident errors are concentrated almost entirely in the dangerous direction.

### Key findings for the memo

**1. SHAP and GBM importance closely agree.** Top 4 features (Lymph nodes, Tumor Size, NPI, Age) identical in both rankings; the next 6 features also identically ranked. SHAP doesn't reveal hidden patterns — it confirms what the GBM's structure already indicates.

**2. The Patient 81 finding from Wisconsin replicates cleanly.** Concentration is statistically indistinguishable between confident-correct (0.856) and confident-wrong (0.854), top-5 feature overlap is 80%. SHAP explanations look the same regardless of whether the underlying prediction is right.

**3. Case studies confirm the qualitative pattern.** All three highest-confidence wrong predictions have favorable clinical profiles — low lymph nodes, low NPI, low grade, Luminal A subtype. SHAP correctly identifies these as protective factors and produces clinically reasonable explanations. The model is making the same prediction a clinician would make based on the same features. It's just wrong about these specific patients, and the explanation can't flag that.

**4. The error pattern is clinically dangerous.** 19 of 22 confident-wrong predictions are false negatives — the model confidently telling patients they're low-risk when they're going to recur. SHAP cannot distinguish these confident-but-wrong predictions from the genuinely-confident-correct ones.

**5. The aggregate SHAP plot is clinically reasonable.** Looking at the beeswarm plot, all features move in clinically expected directions: more lymph nodes → higher recurrence; younger patients → higher recurrence (a known clinical pattern); LumA → protective; LumB → elevated risk. The model's *aggregate logic* is sensible — the issue is that good aggregate logic produces wrong individual predictions ~33% of the time on this data.

### Implications for the memo

This is a substantive negative finding for one slice of paradigm 3 (post-hoc explanation), and it's the kind worth landing precisely:

- **Explanation methods like SHAP are useful for describing model behavior** at the population level. The aggregate feature importance, the directional patterns, the top-feature ranking — all of this is meaningful and clinically interpretable.
- **Explanation methods are NOT useful for validating individual predictions.** A clinician shown a SHAP plot for a specific patient cannot tell from the explanation whether the prediction is right.
- **This strengthens the case for inherently-interpretable methods.** If the model itself is auditable (like GOSDT's 4-leaf tree), you don't need a faithful-but-uninformative explanation layered on top. The GOSDT tree from Phase 4 is a transparent classifier; the GBM with SHAP attached is an opaque classifier with a faithful description of its opacity.

### Files produced

- `results/tables/shap_analysis.csv`: summary metrics
- `results/tables/shap_feature_importance.csv`: top 20 features by mean |SHAP|
- `results/figures/shap_summary.png`: beeswarm plot
- `results/figures/shap_importance_bar.png`: bar plot of mean |SHAP|